In [3]:
import requests, pandas as pd

def klines(symbol="BTCUSDT", n=500, base="https://api.binance.com"):
    out, end = [], None
    while len(out) < n:
        p = {"symbol": symbol, "interval": "15m", "limit": 1000}
        if end: p["endTime"] = end
        r = requests.get(f"{base}/api/v3/klines", params=p, timeout=10).json()
        out = r + out
        end = r[0][0] - 1
    df = pd.DataFrame(out, columns=["t","o","h","l","c","v","ct","qv","n","tb","tq","x"])
    df = df[["t","o","h","l","c","v"]].astype(float)
    df["t"] = pd.to_datetime(df["t"], unit="ms")
    return df.set_index("t").iloc[-n:]

df = klines()
df = df.iloc[:-1]  # drop the still-forming bar
pc = df["c"].shift()
tr = pd.concat([df["h"]-df["l"], (df["h"]-pc).abs(), (df["l"]-pc).abs()], axis=1).max(axis=1)

W = 1200
atr_sma    = tr.rolling(W).mean()
atr_wilder = tr.ewm(alpha=1/W, adjust=False).mean()

print(atr_sma.iloc[-1], atr_wilder.iloc[-1])
print("bps:", 1e4 * atr_sma.iloc[-1] / df["c"].iloc[-1])

nan 95.54271506606881
bps: nan
